<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 158 (delta 76), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.88 MiB | 5.36 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [2]:
!find . -maxdepth 3 -type f | sort

./.config/active_config
./.config/config_sentinel
./.config/configurations/config_default
./.config/default_configs.db
./.config/gce
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/.last_opt_in_prompt.yaml
./.config/.last_survey_prompt.yaml
./.config/.last_update_check.json
./flyrank-ml-internship/AGENTS.md
./flyrank-ml-internship/CLAUDE.md
./flyrank-ml-internship/DATA_USE.md
./flyrank-ml-internship/docs/data-dictionary.md
./flyrank-ml-internship/docs/flyrank-seo-research-march-2026.pdf
./flyrank-ml-internship/docs/intern-free-tooling-guide.md
./flyrank-ml-internship/docs/ml-core-foundation-framework.md
./flyrank-ml-internship/docs/ml-intern-dataset-and-lane-guide.md
./flyrank-ml-internship/.git/config
./flyrank-ml-internship/.git/description
./flyrank-ml-internship/.git/HEAD
./flyrank-ml-internship/.gitignore
./flyrank-ml-internship/.git/index
./flyrank-ml-internship/.git/packed-refs
./flyrank-ml-internship/GUIDE.md
./flyrank-ml-internship/LICENSE
./

In [3]:
!find . -type f | grep -E '\.(csv|json|pkl|joblib|ipynb)$' | sort

./.config/.last_update_check.json
./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
./flyrank-ml-internship/notebooks/01_first_look_and_discovery.ipynb
./flyrank-ml-internship/notebooks/02_your_first_readable_model.ipynb
./flyrank-ml-internship/notebooks/03_working_with_the_full_release.ipynb
./flyrank-ml-internship/outputs/refresh_queue_sample.csv
./flyrank-ml-internship/work/notebooks/capstone.ipynb
./flyrank-ml-internship/work/notebooks/w01_research_question.ipynb
./flyrank-ml-internship/work/notebooks/w02_ml_task_framing.ipynb
./flyrank-ml-internship/work/notebooks/w03_data_contract.ipynb
./flyrank-ml-internship/work/notebooks/w03_feature_leakage_check.ipynb
./flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
./flyrank-ml-internship/work/notebooks/w04_signal_audit.ipynb
./flyrank-ml-internship/work/notebooks/w05_model.ipynb
./flyrank-ml-internship/work/notebooks/w06_validation_audit.ipynb
./flyrank-ml-internship/work/notebooks/w07_action_playbook.ipynb
./s

In [4]:
!find work -maxdepth 3 -type f 2>/dev/null | sort

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The model output is used as a decision-support queue for identifying pages that may need content refresh. Pages are ranked by their predicted likelihood of decline, with higher-risk pages reviewed first.

The main actions are:

1. **Refresh first** — pages with a high predicted risk of decline and supporting evidence from the available content signals.
2. **Review next** — pages with moderate predicted risk where the model indicates possible deterioration but human review is needed before action.
3. **Monitor** — pages with lower predicted risk that do not currently require an immediate content change.

Reason codes are used to explain why an item entered the queue. Examples include declining historical performance, weak content signals, or a combination of risk indicators.

The queue is a prioritization tool, not an automatic publishing system. A human should review the page and supporting evidence before making a content change.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/flyrank-ml-internship")
OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source = ROOT / "outputs" / "refresh_queue_sample.csv"
df = pd.read_csv(source)

required = [
    "final_rank",
    "content_id",
    "final_refresh_score",
    "best_model_probability",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "trend_direction",
    "ctr",
    "impressions_90d",
    "sessions_90d",
    "avg_position"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

queue = df[required].copy()

queue = queue.sort_values(
    by=["final_refresh_score", "final_rank"],
    ascending=[False, True]
).reset_index(drop=True)

queue["priority"] = np.select(
    [
        queue["confidence"].eq("high"),
        queue["confidence"].eq("medium")
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

queue["reason"] = queue["final_reason_codes"].str.replace(
    "|", ", ", regex=False
)

queue.insert(0, "priority_order", range(1, len(queue) + 1))

display(queue.head(20))

export_columns = [
    "priority_order",
    "content_id",
    "final_refresh_score",
    "best_model_probability",
    "confidence",
    "priority",
    "suggested_action",
    "reason",
    "trend_direction",
    "ctr",
    "impressions_90d",
    "sessions_90d",
    "avg_position"
]

queue[export_columns].to_csv(
    OUTPUT_DIR / "w07_ranked_actions.csv",
    index=False
)

print("Exported:")
print(OUTPUT_DIR / "w07_ranked_actions.csv")

,priority_order,final_rank,content_id,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,trend_direction,ctr,impressions_90d,sessions_90d,avg_position,priority,reason
0,1,1,content_1f080331fa2b,81.636697,0.782079,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,down,0.05,12834,66,6.8,HIGH,"declining_with_demand, low_ctr_visible_page, l..."
1,2,2,content_6aa43079fb0c,81.447656,0.788105,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.07,8064,23,3.8,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
2,3,3,content_d6570c51c9bd,81.430346,0.847372,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.00,2498,9,10.1,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
3,4,4,content_72e800a9c214,81.034960,0.774371,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.12,13790,27,8.2,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
4,5,5,content_e04eb9549989,80.873188,0.814805,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.09,3393,5,3.6,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
5,6,6,content_b69288c5e701,80.754770,0.795713,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.07,5811,14,6.4,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
6,7,7,content_9b6df29f7889,80.632923,0.846245,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.12,1622,20,3.1,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
7,8,8,content_bb6ebb5ec8c8,80.371236,0.834638,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.00,2621,10,12.8,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
8,9,9,content_4d76cdb3387b,80.362748,0.843092,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.13,1597,5,2.7,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
9,10,10,content_b4f35d640b1c,80.321757,0.843803,medium,refresh,declining_with_demand|model_decline_risk|visib...,down,0.05,3867,5,27.5,MEDIUM,"declining_with_demand, model_decline_risk, vis..."


Exported:
/content/flyrank-ml-internship/work/outputs/w07_ranked_actions.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended to help content and SEO teams prioritize pages that may need a refresh. The ranked queue uses model predictions together with observable signals such as traffic, CTR, engagement, search position, content age, freshness, and trend direction.

The recommendations are decision-support signals, not automatic publishing decisions. A high-ranked page should be reviewed by a human before any content changes are made.

The model may be less reliable when there is limited traffic, unusual search behavior, missing data, major changes in search intent, or changes in the competitive environment. A recommendation can also become stale as page performance and search demand change.

The output should therefore be used to decide what to investigate first, rather than as proof that a page definitely needs a refresh.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
intended_use = {
    "primary_users": [
        "SEO teams",
        "content teams",
        "content strategists",
        "human reviewers"
    ],
    "primary_use": [
        "Prioritize pages for investigation",
        "Identify possible content refresh candidates",
        "Understand the reasons behind each recommendation"
    ],
    "decision_rule": "Use the ranked queue as decision support and require human review before making content changes.",
    "important_limits": [
        "Predictions depend on the quality and availability of input data.",
        "Low-traffic pages may have less reliable signals.",
        "Search intent and competitive conditions can change.",
        "Model recommendations can become stale over time.",
        "A high score does not prove that a page must be refreshed.",
        "The system should not automatically publish or modify content."
    ]
}

for key, value in intended_use.items():
    print(f"\n{key.upper()}")
    if isinstance(value, list):
        for item in value:
            print(f"- {item}")
    else:
        print(value)


PRIMARY_USERS
- SEO teams
- content teams
- content strategists
- human reviewers

PRIMARY_USE
- Prioritize pages for investigation
- Identify possible content refresh candidates
- Understand the reasons behind each recommendation

DECISION_RULE
Use the ranked queue as decision support and require human review before making content changes.

IMPORTANT_LIMITS
- Predictions depend on the quality and availability of input data.
- Low-traffic pages may have less reliable signals.
- Search intent and competitive conditions can change.
- Model recommendations can become stale over time.
- A high score does not prove that a page must be refreshed.
- The system should not automatically publish or modify content.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a recommendation, a human reviewer should check the page's current search intent, recent performance, CTR, engagement, ranking position, content freshness, and the reason codes produced by the model.

The reviewer should confirm that the recommended action makes sense for the actual page and business context. The model score should be treated as a prioritization signal rather than a final decision.

#### Human review checklist

- Confirm that the page is still relevant to the target audience.
- Check whether the search intent has changed.
- Review recent impressions, CTR, sessions, and average position.
- Check the reason codes behind the recommendation.
- Review the existing page before deciding what should change.
- Check for important business, legal, or factual context.
- Confirm that the proposed refresh is useful to the reader.

#### No-go list

The system should never automatically:

- Publish or overwrite website content.
- Delete a page based only on the model score.
- Change factual, legal, medical, or business-critical information without human review.
- Treat a high model probability as proof that a page is declining.
- Make irreversible content changes without approval.
- Replace human judgment about search intent or business priorities.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
human_review = {
    "checks_before_action": [
        "Confirm current search intent",
        "Review recent impressions, CTR, sessions, and average position",
        "Check content freshness and recent performance",
        "Review the model reason codes",
        "Inspect the actual page before making changes",
        "Consider business, legal, and factual context",
        "Confirm that the proposed refresh benefits the reader"
    ],
    "no_go": [
        "Automatically publish or overwrite content",
        "Delete a page based only on the model score",
        "Change critical factual or business information without review",
        "Treat model probability as proof of decline",
        "Make irreversible changes without approval",
        "Replace human judgment about search intent or business priorities"
    ]
}

print("HUMAN REVIEW CHECKS")
for item in human_review["checks_before_action"]:
    print(f"- {item}")

print("\nNO-GO LIST")
for item in human_review["no_go"]:
    print(f"- {item}")

HUMAN REVIEW CHECKS
- Confirm current search intent
- Review recent impressions, CTR, sessions, and average position
- Check content freshness and recent performance
- Review the model reason codes
- Inspect the actual page before making changes
- Consider business, legal, and factual context
- Confirm that the proposed refresh benefits the reader

NO-GO LIST
- Automatically publish or overwrite content
- Delete a page based only on the model score
- Change critical factual or business information without review
- Treat model probability as proof of decline
- Make irreversible changes without approval
- Replace human judgment about search intent or business priorities


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations should be reviewed periodically because content performance, search demand, and competitive conditions can change.

Monitoring should focus on whether the recommendation quality remains useful and whether the underlying signals have changed significantly.

#### Monitoring triggers

- A sustained change in CTR or impressions.
- A sustained change in average search position.
- Changes in engagement or sessions.
- Significant changes in content freshness.
- Changes in search intent or competitive conditions.
- A noticeable increase in incorrect or unhelpful recommendations.
- Changes in the distribution of model probabilities or confidence levels.

#### Retrain / review trigger

Retraining or model review should be considered when the relationship between the model predictions and observed content decline changes materially, or when recommendation quality consistently deteriorates.

Until the model is reviewed, recommendations should receive additional human scrutiny rather than being treated as automatic decisions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring = {
    "signals_to_monitor": [
        "CTR",
        "Impressions",
        "Average search position",
        "Sessions",
        "Engagement",
        "Content freshness",
        "Search intent",
        "Competitive conditions",
        "Recommendation quality",
        "Model probability and confidence distribution"
    ],
    "review_triggers": [
        "Sustained change in CTR or impressions",
        "Sustained change in average position",
        "Significant engagement or session changes",
        "Major freshness changes",
        "Search intent or competitive changes",
        "Increase in incorrect or unhelpful recommendations",
        "Material change in model probability or confidence patterns"
    ],
    "retrain_trigger": (
        "Consider retraining or model review when the relationship between "
        "model predictions and observed content decline changes materially, "
        "or recommendation quality consistently deteriorates."
    )
}

print("SIGNALS TO MONITOR")
for item in monitoring["signals_to_monitor"]:
    print(f"- {item}")

print("\nREVIEW TRIGGERS")
for item in monitoring["review_triggers"]:
    print(f"- {item}")

print("\nRETRAIN TRIGGER")
print(monitoring["retrain_trigger"])

SIGNALS TO MONITOR
- CTR
- Impressions
- Average search position
- Sessions
- Engagement
- Content freshness
- Search intent
- Competitive conditions
- Recommendation quality
- Model probability and confidence distribution

REVIEW TRIGGERS
- Sustained change in CTR or impressions
- Sustained change in average position
- Significant engagement or session changes
- Major freshness changes
- Search intent or competitive changes
- Increase in incorrect or unhelpful recommendations
- Material change in model probability or confidence patterns

RETRAIN TRIGGER
Consider retraining or model review when the relationship between model predictions and observed content decline changes materially, or recommendation quality consistently deteriorates.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The action playbook produces reusable outputs that can be referenced in the research paper and future analysis.

The ranked action queue is exported as a CSV containing the priority order, content identifier, refresh score, model probability, confidence, suggested action, reason codes, trend information, and supporting performance signals.

The export is stored under `work/outputs/` so that it can be reused without rerunning the full analysis.

The exported queue is intended to support reporting and reproducibility. It should not be treated as a standalone decision artifact without the human-review rules documented in this notebook.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/flyrank-ml-internship")
OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the ranked queue created in Section 1
if "queue" not in globals():
    raise RuntimeError(
        "The ranked queue is not available. Run Section 1 before Section 5."
    )

# --------------------------------------------------
# CSV export
# --------------------------------------------------

csv_path = OUTPUT_DIR / "w07_ranked_actions.csv"

queue.to_csv(
    csv_path,
    index=False
)

# --------------------------------------------------
# Summary export
# --------------------------------------------------

summary = {
    "rows_exported": int(len(queue)),
    "source": "outputs/refresh_queue_sample.csv",
    "score_column": "final_refresh_score",
    "ranking_direction": "descending",
    "human_review_required": True,
    "automated_publishing_allowed": False,
    "purpose": "Content refresh prioritization and decision support"
}

json_path = OUTPUT_DIR / "w07_action_playbook_summary.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("EXPORTS CREATED")
print("-" * 50)

print(f"CSV : {csv_path}")
print(f"JSON: {json_path}")

print("\nRows exported:", len(queue))

print("\nTop 10 actions:")
display(
    queue.head(10)
)

EXPORTS CREATED
--------------------------------------------------
CSV : /content/flyrank-ml-internship/work/outputs/w07_ranked_actions.csv
JSON: /content/flyrank-ml-internship/work/outputs/w07_action_playbook_summary.json

Rows exported: 200

Top 10 actions:


,priority_order,final_rank,content_id,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,trend_direction,ctr,impressions_90d,sessions_90d,avg_position,priority,reason
0,1,1,content_1f080331fa2b,81.636697,0.782079,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,down,0.05,12834,66,6.8,HIGH,"declining_with_demand, low_ctr_visible_page, l..."
1,2,2,content_6aa43079fb0c,81.447656,0.788105,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.07,8064,23,3.8,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
2,3,3,content_d6570c51c9bd,81.430346,0.847372,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.00,2498,9,10.1,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
3,4,4,content_72e800a9c214,81.034960,0.774371,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.12,13790,27,8.2,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
4,5,5,content_e04eb9549989,80.873188,0.814805,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.09,3393,5,3.6,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
5,6,6,content_b69288c5e701,80.754770,0.795713,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.07,5811,14,6.4,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
6,7,7,content_9b6df29f7889,80.632923,0.846245,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.12,1622,20,3.1,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
7,8,8,content_bb6ebb5ec8c8,80.371236,0.834638,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.00,2621,10,12.8,HIGH,"declining_with_demand, low_ctr_visible_page, m..."
8,9,9,content_4d76cdb3387b,80.362748,0.843092,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,0.13,1597,5,2.7,MEDIUM,"declining_with_demand, low_ctr_visible_page, m..."
9,10,10,content_b4f35d640b1c,80.321757,0.843803,medium,refresh,declining_with_demand|model_decline_risk|visib...,down,0.05,3867,5,27.5,MEDIUM,"declining_with_demand, model_decline_risk, vis..."


In [11]:
from pathlib import Path

output_dir = Path("/content/flyrank-ml-internship/work/outputs")

files = sorted(output_dir.iterdir())

print("WORK/OUTPUTS CONTENTS")
print("-" * 40)

for file in files:
    if file.is_file():
        print(f"{file.name:45} {file.stat().st_size:,} bytes")

WORK/OUTPUTS CONTENTS
----------------------------------------
w07_action_playbook_summary.json              294 bytes
w07_ranked_actions.csv                        72,566 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.